# Milestone 1 — User Authentication Module
**Infosys Springboard 7.0 — Intelligent Freight Quote Generation**

Streamlit + JWT + ngrok + Gmail OTP. Run the cells below in order.

Before running: add these four secrets in the Colab Secrets tab (key icon, left sidebar) and enable notebook access for each:
- `JWT_SECRET` — any long random string
- `NGROK_AUTHTOKEN` — from ngrok.com dashboard
- `EMAIL_ADDRESS` — the Gmail address that sends OTPs
- `EMAIL_PASSWORD` — Gmail App Password, no spaces (requires 2-Step Verification enabled)

**Admin login:** username `admin`, password `Admin@123` (hardcoded per spec, not a signup account — change this before submitting if your mentor wants a different value).

**Important:** every time you restart the runtime, the database resets and you must sign up a fresh test account again before testing Forgot Password.


In [ ]:
!pip install -q plotly streamlit pyjwt bcrypt pyngrok

In [ ]:
%%writefile app.py
import plotly.graph_objects as go
import os, re, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.utils import formatdate, make_msgid

# ── Secrets ──
# NOTE: this app runs as a separate background process from the notebook,
# so it CANNOT call google.colab.userdata directly. The launcher cell reads
# the Colab Secrets and passes them in as environment variables instead.
def get_secret(name, default=None):
    return os.getenv(name, default)

JWT_SECRET = get_secret("JWT_SECRET", "dev-only-change-me")
EMAIL_ADDRESS = get_secret("EMAIL_ADDRESS")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD")

# Admin credentials are hardcoded per spec — NOT a signup account, NOT in the users table
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "Admin@123"

SECURITY_QUESTIONS = [
    "What is your pet's name?",
    "What is your mother's maiden name?",
    "What was the name of your first school?",
    "What is your favourite city?",
]

OTP_EXPIRY_MINUTES = 5
DB_NAME = "auth.db"

st.set_page_config(page_title="Freight Quote Auth", page_icon="🔐", layout="centered")

COLORS = {
    "bg": "#0f1117", "card": "#161a26", "border": "#2d3348",
    "accent": "#22c58b", "accent_soft": "#1d3b30", "text": "#e6e8ee", "muted": "#8a8f9c",
}

st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700;800&display=swap');

html,body,.stApp{{
background:linear-gradient(135deg,#071428,#0d2b45,#1d4e89);
font-family:'Poppins',sans-serif;
color:white;
}}

.block-container{{
padding-top:2rem;
padding-bottom:2rem;
}}

h1,h2,h3,h4{{
color:white;
font-weight:700;
}}

[data-testid="stForm"]{{
background:rgba(255,255,255,0.08);
backdrop-filter:blur(18px);
padding:25px;
border-radius:20px;
border:1px solid rgba(255,255,255,.15);
box-shadow:0 10px 30px rgba(0,0,0,.35);
}}

.stTextInput input,
.stSelectbox div[data-baseweb="select"]>div{{
border-radius:12px!important;
background:rgba(255,255,255,.12)!important;
color:white!important;
border:1px solid rgba(255,255,255,.2)!important;
}}

.stButton>button{{
width:100%;
background:linear-gradient(90deg,#00c6ff,#0072ff);
color:white;
font-weight:700;
padding:.7rem;
border-radius:12px;
border:none;
transition:.3s;
}}

.stButton>button:hover{{
transform:translateY(-2px);
box-shadow:0 10px 25px rgba(0,114,255,.45);
}}

div[data-testid="stMetric"]{{
background:rgba(255,255,255,.08);
border-radius:18px;
padding:18px;
border:1px solid rgba(255,255,255,.15);
}}

table{{
border-radius:15px;
overflow:hidden;
}}

footer,header{{
visibility:hidden;
}}
</style>
""",unsafe_allow_html=True)

# ---------------- Database ----------------
def get_db():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def init_db():
    with get_db() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            email TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL
        )""")

init_db()

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    return bcrypt.checkpw(t.encode(), h.encode()) if h else False

# ---------------- Validation ----------------
# At least 2 letters before @, at least 2 letters between @ and dot, at least 2 letters after final dot
EMAIL_RE = re.compile(r"^[A-Za-z]{2,}[\w.+-]*@[A-Za-z]{2,}\.[A-Za-z]{2,}$")

def is_valid_email(email):
    return bool(EMAIL_RE.match(email))

def password_errors(pwd):
    errs = []
    if len(pwd) < 8:
        errs.append("at least 8 characters")
    if not re.search(r"[A-Z]", pwd):
        errs.append("one uppercase letter")
    if not re.search(r"[a-z]", pwd):
        errs.append("one lowercase letter")
    if not re.search(r"[0-9]", pwd):
        errs.append("one number")
    if not re.search(r"[^A-Za-z0-9]", pwd):
        errs.append("one special symbol")
    return errs

# ---------------- JWT (session tokens) ----------------
def make_jwt(username, email):
    payload = {
        "sub": username,
        "email": email,
        "iat": datetime.datetime.utcnow(),
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

# ---------------- OTP (Forgot Password — email route) ----------------
def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    payload = {
        "sub": email,
        "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "iat": datetime.datetime.utcnow(),
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"OTP expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_otp_email(to_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Email is not configured — add EMAIL_ADDRESS and EMAIL_PASSWORD to Colab Secrets."
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Freight Quote Support <{EMAIL_ADDRESS}>"
    msg["To"] = to_email
    msg["Subject"] = "Your password reset OTP"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()

    text_body = f"Your OTP is {otp}. It expires in {OTP_EXPIRY_MINUTES} minutes. If you did not request this, ignore this email."
    html_body = f"""<html><body style="font-family:Arial,sans-serif;">
        <p>We received a request to reset your password.</p>
        <h2 style="letter-spacing:4px;">{otp}</h2>
        <p>This code expires in {OTP_EXPIRY_MINUTES} minutes.</p>
        <p style="color:#888;font-size:12px;">If you did not request this, you can safely ignore this email.</p>
    </body></html>"""
    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    try:
        s = smtplib.SMTP("smtp.gmail.com", 587)
        s.starttls()
        s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        s.quit()
        return True, "OTP sent."
    except Exception as e:
        return False, f"SMTP error: {e}"

# ---------------- Session state ----------------
defaults = {
    "token": None, "page": "Login", "is_admin": False,
    "fp_stage": "start", "fp_email": None, "fp_username": None,
    "fp_question": None, "fp_otp_token": None,
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

def goto(page):
    st.session_state.page = page
    st.rerun()

def reset_fp_state():
    st.session_state.fp_stage = "start"
    st.session_state.fp_email = None
    st.session_state.fp_username = None
    st.session_state.fp_question = None
    st.session_state.fp_otp_token = None

# ================================================================
# UI
# ================================================================
if st.session_state.token or st.session_state.is_admin:
    pass  # header banner is rendered inside each dashboard below
else:
    st.markdown("""
<h1 style='text-align:center;font-size:48px;'>
🚚 Freight Quote AI Portal
</h1>

<p style='text-align:center;
font-size:18px;
color:#d8e6ff;'>
Secure JWT Authentication • OTP Verification • Admin Dashboard
</p>
""",unsafe_allow_html=True)

if st.session_state.token or st.session_state.is_admin:
    # ---------------- Logged-in area ----------------
    if st.session_state.is_admin:
        st.markdown(f"""
        <div style="background:{COLORS['card']};border:1px solid {COLORS['border']};
                    border-radius:14px;padding:20px 24px;margin-bottom:20px;
                    display:flex;justify-content:space-between;align-items:center;">
            <div>
                <div style="font-family:'Poppins',sans-serif;font-size:26px;font-weight:800;">
                    🛡️ Intelligent Freight Quote
                </div>
                <div style="color:{COLORS['muted']};font-size:13px;">Admin Control Panel</div>
            </div>
            <div style="background:{COLORS['accent']};padding:6px 16px;border-radius:20px;
                        font-weight:600;font-size:13px;">🛡️ Administrator</div>
        </div>
        """, unsafe_allow_html=True)

        with get_db() as conn:
            rows = conn.execute("SELECT username, email FROM users").fetchall()

        st.metric("Registered Users", len(rows))
        st.write("")
        if rows:
            st.table({"Username": [r[0] for r in rows], "Email": [r[1] for r in rows]})
        else:
            st.info("No users registered yet.")

        if st.button("Logout"):
            st.session_state.is_admin = False
            goto("Login")
    else:
        payload = verify_jwt(st.session_state.token)
        with st.sidebar:
            st.image(
                "https://img.icons8.com/fluency/96/cargo-ship.png",
                width=80
            )

            st.markdown("## Freight Quote AI")

            menu = st.radio(
              "Navigation",
              [
                  "🏠 Dashboard",
                  "👤 Profile",
                  "📊 Analytics",
                  "🔐 Security"
              ]
          )

        st.markdown("---")

        if st.button("🚪 Logout", use_container_width=True):
              st.session_state.token = None
              goto("Login")

        st.markdown("---")
        st.caption("Infosys Springboard\nMilestone 1")


        if not payload:
            st.session_state.token = None
            goto("Login")
        else:
            st.markdown(f"""
            <div style="background:{COLORS['card']};border:1px solid {COLORS['border']};
                        border-radius:14px;padding:20px 24px;margin-bottom:20px;
                        display:flex;justify-content:space-between;align-items:center;">
                <div>
                    <div style="font-family:'Poppins',sans-serif;font-size:26px;font-weight:800;">
                        🔐 Intelligent Freight Quote
                    </div>
                    <div style="color:{COLORS['muted']};font-size:13px;">Account Dashboard</div>
                </div>
                <div style="background:{COLORS['accent']};padding:6px 16px;border-radius:20px;
                            font-weight:600;font-size:13px;">👤 {payload['sub']}</div>
            </div>
            """, unsafe_allow_html=True)

            st.markdown(f"""
            <div style="
            background:#10253d;
            padding:20px;
            border-radius:18px;
            margin-bottom:25px;
            border-left:6px solid #00d4ff;
            ">
            <h2>👋 Welcome, {payload['sub']}</h2>

            <p>{payload['email']}</p>

            </div>
            """, unsafe_allow_html=True)

            exp_time_utc = datetime.datetime.utcfromtimestamp(payload['exp'])
            exp_time_ist = exp_time_utc + datetime.timedelta(hours=5, minutes=30)
            exp_display = exp_time_ist.strftime("%I:%M %p IST")
            gap1,c1,c2,c3,c4,gap2 = st.columns([0.2,1,1,1,1,0.2])
            c1.markdown("""
            <div style="background:#16263d;padding:14px;border-radius:15px;text-align:center;">
            <div style="font-size:30px;">🔐</div>
            <div style="font-size:13px;color:#8db7ff;">AUTH METHOD</div>
            <div style="font-size:24px;font-weight:700;">JWT</div>
            <div style="font-size:11px;color:gray;">JSON Web Token</div>
            </div>
            """, unsafe_allow_html=True)

            c2.markdown(f"""
            <div style="
            background:#16263d;
            padding:12px;
            border-radius:15px;
            text-align:center;
            ">

            <div style="
            width:48px;
            height:48px;
            margin:auto;
            border-radius:50%;
            background:#5b3b99;
            display:flex;
            align-items:center;
            justify-content:center;
            font-size:24px;
            margin-bottom:6px;
            ">
            ⏰
            </div>

            <div style="font-size:11px;color:#8db7ff;">
            SESSION
            </div>

            <div style="font-size:20px;font-weight:700;">
            {exp_display}
            </div>

            <div style="font-size:10px;color:gray;">
            Expires Today
            </div>

            </div>
            """, unsafe_allow_html=True)

            c3.markdown("""
            <div style="background:#16263d;padding:14px;border-radius:15px;text-align:center;">
            <div style="font-size:30px;">🛡️</div>
            <div style="font-size:13px;color:#8db7ff;">STATUS</div>
            <div style="font-size:22px;font-weight:700;color:#47ff8d;">Secure</div>
            <div style="font-size:11px;color:gray;">Session Active</div>
            </div>
            """, unsafe_allow_html=True)

            c4.markdown("""
            <div style="background:#16263d;padding:14px;border-radius:15px;text-align:center;">
            <div style="font-size:30px;">📧</div>
            <div style="font-size:13px;color:#8db7ff;">EMAIL</div>
            <div style="font-size:22px;font-weight:700;">Verified</div>
            <div style="font-size:11px;color:gray;">Confirmed</div>
            </div>
            """, unsafe_allow_html=True)
            st.markdown("### 📊 Security Analytics")

            fig = go.Figure(go.Indicator(
                mode="gauge+number",
                value=95,
                title={"text":"Security Score"},
                gauge={
                    "axis":{"range":[0,100]},
                    "bar":{"color":"#00D4FF"},
                    "steps":[
                        {"range":[0,50],"color":"#8B0000"},
                        {"range":[50,80],"color":"#DAA520"},
                        {"range":[80,100],"color":"#0B6623"},
                    ]
                }
            ))

            fig.update_layout(
                height=320,
                paper_bgcolor="rgba(0,0,0,0)",
                font_color="white"
            )

            st.plotly_chart(fig, use_container_width=True)
            st.markdown("### 📜 Recent Activity")

            st.success("✅ Login Successful")

            st.info("🔐 JWT Token Generated")

            st.info("📧 Email Verified")

            st.success("🛡 Secure Session Active")
            st.markdown("### 👤 User Profile")

            col1, col2 = st.columns(2)

            with col1:
                st.write("**Username**")
                st.success(payload["sub"])

                st.write("**Email**")
                st.success(payload["email"])

            with col2:
                st.write("**Authentication**")
                st.success("JWT")

                st.write("**Recovery**")
                st.success("Email OTP / Security Question")

            st.write("")


else:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    # ---------------- LOGIN ----------------
    if st.session_state.page == "Login":
        st.subheader("Sign In")
        with st.form("login_form"):
            identifier = st.text_input("Username or Email")
            password = st.text_input("Password", type="password")
            submitted = st.form_submit_button("Sign In")

        if submitted:
            if not identifier or not password:
                st.error("Please fill in all fields.")
            elif identifier == ADMIN_USERNAME and password == ADMIN_PASSWORD:
                st.session_state.is_admin = True
                st.rerun()
            else:
                with get_db() as conn:
                    row = conn.execute(
                        "SELECT username, email, password_hash FROM users WHERE username=? OR email=?",
                        (identifier, identifier)
                    ).fetchone()
                if row and check_txt(password, row[2]):
                    st.session_state.token = make_jwt(row[0], row[1])
                    st.rerun()
                else:
                    st.error("Invalid username/email or password.")  # generic — doesn't reveal which was wrong

        c1, c2 = st.columns(2)
        if c1.button("Create an account", use_container_width=True):
            goto("Signup")
        if c2.button("Forgot password?", use_container_width=True):
            goto("Forgot")

    # ---------------- SIGNUP ----------------
    elif st.session_state.page == "Signup":
        st.subheader("Create Account")
        with st.form("signup_form"):
            username = st.text_input("Username")
            email = st.text_input("Email")
            password = st.text_input("Password", type="password")
            confirm = st.text_input("Confirm Password", type="password")
            question = st.selectbox("Security Question", SECURITY_QUESTIONS)
            answer = st.text_input("Security Answer")
            submitted = st.form_submit_button("Sign Up")

        if submitted:
            if not all([username, email, password, confirm, answer]):
                st.error("All fields are required.")
            elif not is_valid_email(email):
                st.error("Please enter a valid email address (e.g. ab@cd.ef).")
            else:
                errs = password_errors(password)
                if errs:
                    st.error("Password must contain " + ", ".join(errs) + ".")
                elif password != confirm:
                    st.error("Passwords do not match.")
                else:
                    try:
                        with get_db() as conn:
                            conn.execute(
                                "INSERT INTO users (username, email, password_hash, security_question, security_answer_hash) VALUES (?, ?, ?, ?, ?)",
                                (username.strip(), email.lower().strip(), hash_txt(password),
                                 question, hash_txt(answer.lower().strip()))
                            )
                        st.success("Account created! Please log in.")
                        time.sleep(1)
                        goto("Login")
                    except sqlite3.IntegrityError:
                        st.error("That username or email is already registered.")

        if st.button("← Back to Login"):
            goto("Login")

    # ---------------- FORGOT PASSWORD ----------------
    elif st.session_state.page == "Forgot":
        st.subheader("Reset Password")

        if st.session_state.fp_stage == "start":
            method = st.radio("Choose a recovery method", ["Security Question", "Email OTP"])
            identifier = st.text_input(
                "Username (for Security Question) or Email (for OTP)"
            )
            if st.button("Continue"):
                if not identifier:
                    st.error("Please enter your username or email.")
                elif method == "Security Question":
                    with get_db() as conn:
                        row = conn.execute(
                            "SELECT username, security_question FROM users WHERE username=? OR email=?",
                            (identifier, identifier)
                        ).fetchone()
                    if row:
                        st.session_state.fp_username = row[0]
                        st.session_state.fp_question = row[1]
                        st.session_state.fp_stage = "sq_answer"
                        st.rerun()
                    else:
                        st.error("No matching account found.")
                else:
                    with get_db() as conn:
                        row = conn.execute(
                            "SELECT email FROM users WHERE email=?", (identifier.lower().strip(),)
                        ).fetchone()
                    if row:
                        otp = generate_otp()
                        ok, msg = send_otp_email(row[0], otp)
                        if ok:
                            st.session_state.fp_email = row[0]
                            st.session_state.fp_otp_token = make_otp_token(row[0], otp)
                            st.session_state.fp_stage = "otp_verify"
                            st.success("OTP sent to your email.")
                            time.sleep(1)
                            st.rerun()
                        else:
                            st.error(msg)
                    else:
                        st.error("No account found with that email.")

        elif st.session_state.fp_stage == "sq_answer":
            st.info(f"❓ {st.session_state.fp_question}")
            answer = st.text_input("Your Answer")
            new_pw = st.text_input("New Password", type="password")
            confirm_pw = st.text_input("Confirm New Password", type="password")
            if st.button("Reset Password"):
                if not all([answer, new_pw, confirm_pw]):
                    st.error("All fields are required.")
                else:
                    errs = password_errors(new_pw)
                    if errs:
                        st.error("Password must contain " + ", ".join(errs) + ".")
                    elif new_pw != confirm_pw:
                        st.error("Passwords do not match.")
                    else:
                        with get_db() as conn:
                            row = conn.execute(
                                "SELECT security_answer_hash FROM users WHERE username=?",
                                (st.session_state.fp_username,)
                            ).fetchone()
                        if row and check_txt(answer.lower().strip(), row[0]):
                            with get_db() as conn:
                                conn.execute(
                                    "UPDATE users SET password_hash=? WHERE username=?",
                                    (hash_txt(new_pw), st.session_state.fp_username)
                                )
                            st.success("Password reset! Please log in.")
                            time.sleep(1)
                            reset_fp_state()
                            goto("Login")
                        else:
                            st.error("Incorrect security answer.")

        elif st.session_state.fp_stage == "otp_verify":
            st.info(f"📧 Enter the OTP sent to {st.session_state.fp_email}")
            otp_input = st.text_input("6-digit OTP", max_chars=6)
            if st.button("Verify OTP"):
                ok, msg = verify_otp_token(
                    st.session_state.fp_otp_token, otp_input, st.session_state.fp_email
                )
                if ok:
                    st.session_state.fp_stage = "otp_reset"
                    st.rerun()
                else:
                    st.error(msg)

        elif st.session_state.fp_stage == "otp_reset":
            new_pw = st.text_input("New Password", type="password")
            confirm_pw = st.text_input("Confirm New Password", type="password")
            if st.button("Update Password"):
                errs = password_errors(new_pw)
                if errs:
                    st.error("Password must contain " + ", ".join(errs) + ".")
                elif new_pw != confirm_pw:
                    st.error("Passwords do not match.")
                else:
                    with get_db() as conn:
                        conn.execute(
                            "UPDATE users SET password_hash=? WHERE email=?",
                            (hash_txt(new_pw), st.session_state.fp_email)
                        )
                    st.success("Password updated! Please log in.")
                    time.sleep(1)
                    reset_fp_state()
                    goto("Login")

        if st.button("← Cancel"):
            reset_fp_state()
            goto("Login")


In [ ]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

# ── Read secrets HERE, in the notebook, where Colab Secrets are accessible ──
env = os.environ.copy()
env["JWT_SECRET"] = userdata.get("JWT_SECRET")
env["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
env["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")

NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing tunnels / streamlit sessions
ngrok.kill()
!pkill -f streamlit
time.sleep(2)

# Start Streamlit in the background, PASSING the secrets in as env vars
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"Live URL: {public_url}")
print("=" * 60)
print("Press the Colab Stop button (or Ctrl+C) to shut down.")
print("This cell will keep 'running' the whole time your app is live — that is normal, not stuck.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("Stopped.")
